# Cancellations — simple version

For the original log and each policy we compute, per simulated run, then average over runs:

- **avg_events_per_case**: mean number of events in a case.
- **p25 / p50 / p75_events**: 25th, 50th (median) and 75th percentile of the number of events per case.
- **cancelled_%**: % of cases whose **last** activity is a cancellation activity.
- **early_cancelled_%**: % of cases with 3 or fewer events (any activities, e.g. `ABC` yes, `ABCD` no).

Only AcademicCredentials has cancellation activities, so `cancelled_%` is empty (NaN) for the BPIC logs.

In [ ]:
import glob
import pandas as pd

LOGS = ["AcademicCredentials", "BPIC_2012", "BPIC_2017"]
CONDITION = "flex"   # folder inside data/evaluation_results/
POLICIES = ["RA-RR", "DM-RR", "DM-GR", "DM-DRL", "DRL-DRL"]
CANCEL_ACTIVITIES = {
    "AcademicCredentials": ["Notificacion estudiante cancelacion soli", "Cancelar Solicitud", "Cancelar curso"],
    "BPIC_2012": [],   # no cancellation activities
    "BPIC_2017": [],   # no cancellation activities
}
EARLY_MAX_EVENTS = 3

In [ ]:
def summarize_cases(csv_path):
    """One row per case: number of events and last activity (by end_time)."""
    events = pd.read_csv(csv_path)
    events["activity"] = events["activity"].str.strip()
    events = events.sort_values(["case_id", "end_time", "start_time"])
    by_case = events.groupby("case_id")
    return pd.DataFrame({"n_events": by_case.size(), "last_activity": by_case["activity"].last()})


def compute_outcomes(cases, cancel_activities):
    is_early = cases["n_events"] <= EARLY_MAX_EVENTS
    outcomes = {
        "avg_events_per_case": cases["n_events"].mean(),
        "p25_events": cases["n_events"].quantile(0.25),
        "p50_events": cases["n_events"].quantile(0.50),
        "p75_events": cases["n_events"].quantile(0.75),
        "cancelled_%": float("nan"),
    }
    if cancel_activities:
        outcomes["cancelled_%"] = 100 * cases["last_activity"].isin(cancel_activities).mean()
    outcomes["early_cancelled_%"] = 100 * is_early.mean()
    return outcomes


def average_over_runs(csv_paths, cancel_activities):
    runs = [compute_outcomes(summarize_cases(path), cancel_activities) for path in csv_paths]
    return pd.DataFrame(runs).mean()

In [ ]:
rows = []
for log in LOGS:
    original_file = [f"../../data/logs/{log}/{log}_train.csv"]
    rows.append({"log": log, "policy": "Original (train)", **average_over_runs(original_file, CANCEL_ACTIVITIES[log])})

    for policy in POLICIES:
        run_files = sorted(glob.glob(f"../../data/evaluation_results/{CONDITION}/{log}/{policy}/simulated_logs/sim_run_*.csv"))
        if not run_files:
            print(f"[skip] {log}/{policy}: no simulated logs")
            continue
        rows.append({"log": log, "policy": policy, **average_over_runs(run_files, CANCEL_ACTIVITIES[log])})

table = pd.DataFrame(rows).set_index(["log", "policy"]).round(2)
print(table.to_string())

                                      avg_events_per_case  p25_events  p50_events  p75_events  cancelled_%  early_cancelled_%
log                 policy                                                                                                   
AcademicCredentials Original (train)                 4.89         4.0         4.0         6.0        54.52               8.04
                    RA-RR                            6.96         4.0         6.0         8.1        60.58               0.00
                    DM-RR                            5.18         4.0         4.0         6.0        50.98               0.00
                    DM-GR                            5.21         4.0         4.0         6.0        50.30               0.00
                    DM-DRL                           5.23         4.0         4.0         6.0        49.12               0.00
                    DRL-DRL                          4.00         4.0         4.0         4.0        99.97            

## DRL-DRL restricted

Same metrics for DRL-DRL in the restriction scenarios (`data/evaluation_results_restricted/<scenario>/<log>/DRL-DRL/`).

In [ ]:
restricted_rows = []
for log in LOGS:
    for scenario_folder in sorted(glob.glob(f"../../data/evaluation_results_restricted/*/{log}/DRL-DRL")):
        scenario = scenario_folder.replace("\\", "/").split("/")[-3]
        run_files = sorted(glob.glob(f"{scenario_folder}/simulated_logs/sim_run_*.csv"))
        restricted_rows.append({"log": log, "scenario": scenario, **average_over_runs(run_files, CANCEL_ACTIVITIES[log])})

restricted_table = pd.DataFrame(restricted_rows).set_index(["log", "scenario"]).round(2)
print(restricted_table.to_string())

                                                         avg_events_per_case  p25_events  p50_events  p75_events  cancelled_%  early_cancelled_%
log                 scenario                                                                                                                    
AcademicCredentials AC_R_limit_cancel                                   4.83         4.0         4.0        5.95        71.61               0.00
BPIC_2012           BPIC12_R1_no_end_after_leads                        3.18         1.0         4.0        4.00          NaN              33.76
BPIC_2017           BPIC17_R_validate_after_call_offers                 2.36         2.0         2.0        3.00          NaN              94.74
